In [7]:
import subprocess, sys
packages = [
    'python-dotenv', 'requests', 'pandas', 'numpy', 
    'datasets', 'rouge-score', 'sentence-transformers', 'ipywidgets'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
import os, re, time
import requests as http_requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from datasets import load_dataset
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util as st_util
from sklearn.model_selection import train_test_split
import ipywidgets as widgets
from IPython.display import display, HTML
load_dotenv()
API_KEY = os.getenv('OPENROUTER_API_KEY')


In [8]:
def degrade_prompt(good_prompt):
    text = good_prompt.lower().strip()
    text = re.sub(r'(act as|you are|as a|as an|i want you to)\s+\w+', '', text)
    text = re.sub(r'[\-\*•]\s+', '', text)
    text = re.sub(r'\d+\.\s+', '', text)
    text = re.sub(r'(in bullet points|step by step|in detail|please|provide|explain)', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    return ' '.join(words[:min(8, max(3, len(words) // 4))])

hf_data = load_dataset('fka/awesome-chatgpt-prompts', split='train')
source_a = pd.DataFrame({
    'weak_prompt': [degrade_prompt(row['prompt']) for row in hf_data],
    'improved_prompt': [row['prompt'] for row in hf_data],
    'source': 'huggingface'
})


In [9]:
SEED_TOPICS = ['explain ai', 'python basics', 'climate change', 'healthy eating']
SYSTEM_PROMPT = "You are an expert AI prompt engineer. Transform the following weak prompt into a highly effective, structured prompt. Include: a clear ROLE, specific ACTION, relevant CONTEXT, desired OUTPUT FORMAT, and SPECIFICITY. Return ONLY the improved prompt text."
OPENROUTER_MODEL = 'mistralai/mistral-small-creative'

def call_openrouter(prompt, system=SYSTEM_PROMPT, max_tokens=300):
    try:
        resp = http_requests.post(
            'https://openrouter.ai/api/v1/chat/completions',
            headers={'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'},
            json={'model': OPENROUTER_MODEL, 'messages': [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}], 'max_tokens': max_tokens, 'temperature': 0.7},
            timeout=30
        )
        resp.raise_for_status()
        return resp.json()['choices'][0]['message']['content'].strip()
    except Exception:
        return None

source_b_rows = []
for topic in SEED_TOPICS:
    improved = call_openrouter(topic)
    if improved and len(improved.split()) >= 15:
        source_b_rows.append({'weak_prompt': topic, 'improved_prompt': improved, 'source': 'openrouter'})
    time.sleep(1.0)

source_b = pd.DataFrame(source_b_rows)
df = pd.concat([source_a, source_b], ignore_index=True)


In [10]:
df.dropna(subset=['weak_prompt', 'improved_prompt'], inplace=True)
df.drop_duplicates(subset=['weak_prompt'], inplace=True)

def clean_text(text):
    text = re.sub(r'<[^>]+>', '', str(text))
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df['weak_prompt'] = df['weak_prompt'].apply(lambda x: clean_text(x).lower())
df['improved_prompt'] = df['improved_prompt'].apply(clean_text)
df['weak_len'] = df['weak_prompt'].apply(lambda x: len(x.split()))
df['improved_len'] = df['improved_prompt'].apply(lambda x: len(x.split()))

df = df[(df['weak_len'] >= 2) & (df['weak_len'] <= 10) & (df['improved_len'] >= 15) & (df['improved_len'] <= 300)].copy()
df = df.sample(min(20, len(df)), random_state=42) # Minimal dataset for quick running


Executing <Task pending name='Task-2' coro=<Kernel.poll_control_queue() running at C:\Users\ankit\AppData\Roaming\Python\Python313\site-packages\ipykernel\kernelbase.py:304> wait_for=<Future pending cb=[Task.task_wakeup()] created at C:\Users\ankit\AppData\Roaming\Python\Python313\site-packages\tornado\queues.py:248> cb=[_chain_future.<locals>._call_set_state() at c:\Users\ankit\anaconda3\Lib\asyncio\futures.py:391] created at c:\Users\ankit\anaconda3\Lib\asyncio\tasks.py:748> took 0.191 seconds


In [11]:
def score_prompt(prompt):
    text = str(prompt).strip()
    if not text: return 0
    words = text.split()
    score = 0
    if len(words) >= 20: score += 2
    elif len(words) >= 10: score += 1
    role_kw = ['act as', 'you are', 'as a', 'as an', 'role:', 'persona:']
    if any(k in text.lower() for k in role_kw): score += 2
    actions = ['explain', 'describe', 'list', 'compare', 'write', 'create', 'summarize', 'analyze', 'generate', 'give me', 'provide', 'show']
    if '?' in text or any(v in text.lower() for v in actions): score += 2
    else: score += 1
    fmt = ['bullet', 'list', 'step', 'table', 'json', 'markdown', 'numbered', 'format', 'structure', 'outline', 'paragraph']
    if any(f in text.lower() for f in fmt): score += 2
    has_nums = bool(re.search(r'\d', text))
    has_proper = bool(re.search(r'[A-Z]', text[1:])) if len(text) > 1 else False
    if has_nums or (has_proper and len(words) >= 15): score += 2
    elif len(words) >= 15: score += 1
    return score


In [12]:
api_outputs = []
for _, row in df.iterrows():
    result = call_openrouter(row['weak_prompt'])
    api_outputs.append(result if result else '')
    time.sleep(1.0)

df['api_output'] = api_outputs
valid_mask = df['api_output'].str.len() > 0
df = df[valid_mask].copy()

df['score_original'] = df['weak_prompt'].apply(score_prompt)
df['score_api_output'] = df['api_output'].apply(score_prompt)
df.to_csv('results.csv', index=False)


In [13]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge_results = {'rouge1': [], 'rouge2': [], 'rougeL': []}
for _, row in df.iterrows():
    scores = scorer.score(row['improved_prompt'], row['api_output'])
    for key in rouge_results:
        rouge_results[key].append(scores[key].fmeasure)

sem_model = SentenceTransformer('all-MiniLM-L6-v2')
gt_embeddings = sem_model.encode(df['improved_prompt'].tolist())
api_embeddings = sem_model.encode(df['api_output'].tolist())
cos_sims = [float(st_util.cos_sim(gt_embeddings[i], api_embeddings[i])) for i in range(len(gt_embeddings))]
df['semantic_similarity'] = cos_sims
df.to_csv('report.csv', index=False)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
prompt_input = widgets.Textarea(value='', placeholder='Type a weak prompt...', description='Prompt:', layout=widgets.Layout(width='90%', height='80px'))
output_area = widgets.Output()
def on_optimize(btn):
    output_area.clear_output()
    weak = prompt_input.value.strip()
    if not weak:
        return
    with output_area:
        improved = call_openrouter(weak)
        if not improved: return
        before = score_prompt(weak)
        after = score_prompt(improved)
        display(HTML(f"<div style='background:#1e1b4b; padding:16px; border-radius:12px; color:white;'><p><strong>Original (Score: {before}/10)</strong>: {weak}</p><p><strong>Optimized (Score: {after}/10)</strong>: {improved}</p></div>"))

optimize_btn = widgets.Button(description='🚀 Optimize', button_style='info')
optimize_btn.on_click(on_optimize)
display(prompt_input, optimize_btn, output_area)


Textarea(value='', description='Prompt:', layout=Layout(height='80px', width='90%'), placeholder='Type a weak …

Button(button_style='info', description='🚀 Optimize', style=ButtonStyle())

Output()